In [ ]:
# ============================================================
# WGAN-GP for SELFIES (Discrete) using Gumbel-Softmax (Straight-Through)
# - Same architecture, light stability fixes only
# - Critic trained with Wasserstein loss + Gradient Penalty -- 300 Generation
# - Eval every epoch
# - Best checkpoint + corrected early stopping
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, re, json
import numpy as np
import pandas as pd
from collections import Counter

import tensorflow as tf
from tensorflow.keras import layers, Model

!pip -q install selfies
import selfies as sf

# -------------------------
# 0) GPU stability
# -------------------------
tf.keras.backend.clear_session()
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("✅ GPU memory growth enabled:", gpus)
else:
    print("⚠️ No GPU detected")

# -------------------------
# 1) Paths & Config
# -------------------------
data_path  = '/content/drive/MyDrive/Diss/tokenized_selfies_new.csv'
vocab_path = '/content/drive/MyDrive/Diss/vocab_selfies_new.csv'
out_dir    = '/content/drive/MyDrive/Diss/'
ckpt_dir   = os.path.join(out_dir, "ckpt_wgan_gp_SELFIES_stable")
best_dir   = os.path.join(out_dir, "best_wgan_gp_SELFIES_stable")
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(best_dir, exist_ok=True)

# Data
N_TRAIN = 150_000

# Dataset
BATCH_SIZE = 64
BUFFER_SIZE = 30000

# WGAN Training
WGAN_EPOCHS = 40
STEPS_PER_EPOCH = 200
NOISE_DIM = 128

# Gumbel-Softmax temp (less aggressive annealing)
GUMBEL_TAU_START = 1.2
GUMBEL_TAU_END   = 0.9

# WGAN params (lighter critic pressure)
N_CRITIC = 3
LAMBDA_GP = 5.0

# Optimizers (generator slightly stronger)
g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.0, beta_2=0.9)
d_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)

# Eval
EVAL_EVERY = 1
EVAL_SAMPLES = 300
EVAL_TEMPERATURE = 0.8
EVAL_TOP_K = 60
EVAL_TOP_P = 0.92

# Early stopping
PATIENCE = 5
MIN_DELTA = 1e-6

# -------------------------
# 2) Load Data & Vocab
# -------------------------
df = pd.read_csv(data_path).fillna(0).astype(int)
df = df.sample(n=min(N_TRAIN, len(df)), random_state=42)
X = df.values
seq_len = X.shape[1]

vocab_df = pd.read_csv(vocab_path).sort_values("index").reset_index(drop=True)
idx2token = dict(zip(vocab_df['index'].astype(int), vocab_df['token'].astype(str)))
token2idx = dict(zip(vocab_df['token'].astype(str), vocab_df['index'].astype(int)))

PAD_TOKEN, BOS_TOKEN, EOS_TOKEN = "<pad>", "<bos>", "<eos>"
pad_id = token2idx[PAD_TOKEN]
bos_id = token2idx[BOS_TOKEN]
eos_id = token2idx[EOS_TOKEN]
vocab_size = len(token2idx)

print("X:", X.shape, "seq_len:", seq_len, "vocab:", vocab_size)
print("IDs:", {"PAD": pad_id, "BOS": bos_id, "EOS": eos_id})

ds = tf.data.Dataset.from_tensor_slices(X)
ds = ds.shuffle(BUFFER_SIZE, reshuffle_each_iteration=True)\
       .batch(BATCH_SIZE, drop_remainder=True)\
       .repeat()\
       .prefetch(tf.data.AUTOTUNE)

# -------------------------
# 3) Validity
# -------------------------
allowed_chars_re = re.compile(r"^[A-Za-z0-9@\+\-\=\#\$\%\.\(\)\[\]\\/]+$")
atom_re = re.compile(r"[BCNOFPSIHbcnops]|\[")

def indices_to_selfies(seq):
    out = []
    for idx in seq:
        idx = int(idx)
        if idx in (pad_id, eos_id):
            break
        if idx == bos_id:
            continue
        out.append(idx2token.get(idx, ""))
    return "".join(out)

def selfies_to_smiles(selfies_str):
    try:
        return sf.decoder(str(selfies_str))
    except Exception:
        return ""

def is_smiles_syntax_valid(sm):
    sm = str(sm).strip()
    if not sm or not allowed_chars_re.match(sm) or not atom_re.search(sm):
        return False
    if sm[0] in "=#)" or sm[-1] in "=#(":
        return False
    paren = bracket = 0
    for ch in sm:
        if ch == '(':
            paren += 1
        elif ch == ')':
            paren -= 1
            if paren < 0:
                return False
        elif ch == '[':
            bracket += 1
        elif ch == ']':
            bracket -= 1
            if bracket < 0:
                return False
    if paren != 0 or bracket != 0:
        return False
    digits = re.findall(r"\d", sm)
    for v in Counter(digits).values():
        if v % 2 != 0:
            return False
    return True

def is_selfies_valid(selfies_str):
    sm = selfies_to_smiles(selfies_str)
    if not sm:
        return False
    return is_smiles_syntax_valid(sm)

# -------------------------
# 4) Models: Generator + Critic
# -------------------------
class SeqGenerator(Model):
    def __init__(self, vocab_size, noise_dim=128, hidden=256):
        super().__init__()
        self.fc = layers.Dense(hidden, activation="relu")
        self.rep = layers.RepeatVector(seq_len)
        self.gru = layers.GRU(hidden, return_sequences=True)
        self.out = layers.Dense(vocab_size)

    def call(self, z, training=False):
        x = self.fc(z)
        x = self.rep(x)
        x = self.gru(x, training=training)
        return self.out(x)

class Critic(Model):
    def __init__(self, vocab_size, emb_dim=64, hidden=128, drop=0.3):
        super().__init__()
        self.emb = layers.Embedding(vocab_size, emb_dim)
        self.drop1 = layers.Dropout(drop)
        self.gru = layers.GRU(hidden)
        self.drop2 = layers.Dropout(drop)
        self.out = layers.Dense(1)

    def call(self, x, training=False):
        if len(x.shape) == 3:
            W = self.emb.weights[0]
            x = tf.matmul(x, W)
        else:
            x = self.emb(x)
        x = self.drop1(x, training=training)
        x = self.gru(x, training=training)
        x = self.drop2(x, training=training)
        return self.out(x)

G = SeqGenerator(vocab_size, noise_dim=NOISE_DIM)
D = Critic(vocab_size)

_ = G(tf.zeros((2, NOISE_DIM), dtype=tf.float32), training=False)
_ = D(tf.zeros((2, seq_len), dtype=tf.int32), training=False)

# -------------------------
# 5) Gumbel-Softmax ST
# -------------------------
def _sample_gumbel(shape, eps=1e-20):
    U = tf.random.uniform(shape, minval=0, maxval=1)
    return -tf.math.log(-tf.math.log(U + eps) + eps)

def gumbel_softmax_sample(logits, tau, hard=True):
    g = _sample_gumbel(tf.shape(logits))
    y = tf.nn.softmax((logits + g) / tau, axis=-1)

    if not hard:
        return y

    y_hard = tf.one_hot(tf.argmax(y, axis=-1), depth=tf.shape(y)[-1], dtype=y.dtype)
    y_st = tf.stop_gradient(y_hard - y) + y
    return y_st

# -------------------------
# 6) Checkpoint + Resume
# -------------------------
ckpt = tf.train.Checkpoint(
    G=G,
    D=D,
    g_opt=g_opt,
    d_opt=d_opt,
    epoch=tf.Variable(0, dtype=tf.int64),
    best_valid=tf.Variable(-1.0, dtype=tf.float32),
    patience_counter=tf.Variable(0, dtype=tf.int64),
)
manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=20)

if manager.latest_checkpoint:
    ckpt.restore(manager.latest_checkpoint).expect_partial()
    print("✅ Restored:", manager.latest_checkpoint)
    print("✅ Resuming from epoch:", int(ckpt.epoch.numpy()))
    print("✅ Best validity so far:", float(ckpt.best_valid.numpy()))
else:
    print("🚀 Training from scratch")

# -------------------------
# 7) Gradient Penalty
# -------------------------
def gradient_penalty(real_ids, fake_onehot):
    real_onehot = tf.one_hot(tf.cast(real_ids, tf.int32), depth=vocab_size, dtype=tf.float32)
    alpha = tf.random.uniform([tf.shape(real_onehot)[0], 1, 1], 0.0, 1.0)
    x_hat = alpha * real_onehot + (1.0 - alpha) * fake_onehot

    with tf.GradientTape() as tape:
        tape.watch(x_hat)
        d_hat = D(x_hat, training=True)
    grads = tape.gradient(d_hat, x_hat)
    grads = tf.reshape(grads, [tf.shape(grads)[0], -1])
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    gp = tf.reduce_mean((norm - 1.0) ** 2)
    return gp

# -------------------------
# 8) WGAN train step
# -------------------------
@tf.function
def critic_step(real_batch_ids, tau):
    bsz = tf.shape(real_batch_ids)[0]
    z = tf.random.normal([bsz, NOISE_DIM])
    fake_logits = G(z, training=True)
    fake_onehot = gumbel_softmax_sample(fake_logits, tau)

    with tf.GradientTape() as tape:
        d_real = D(tf.cast(real_batch_ids, tf.int32), training=True)
        d_fake = D(fake_onehot, training=True)

        w_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real)
        gp = gradient_penalty(real_batch_ids, fake_onehot)
        d_loss = w_loss + LAMBDA_GP * gp

    grads = tape.gradient(d_loss, D.trainable_variables)
    grads = [tf.clip_by_norm(g, 5.0) if g is not None else None for g in grads]
    d_opt.apply_gradients(zip(grads, D.trainable_variables))
    return d_loss

@tf.function
def generator_step(tau):
    bsz = tf.constant(BATCH_SIZE, dtype=tf.int32)
    z = tf.random.normal([bsz, NOISE_DIM])

    with tf.GradientTape() as tape:
        fake_logits = G(z, training=True)
        fake_onehot = gumbel_softmax_sample(fake_logits, tau)
        d_fake = D(fake_onehot, training=True)
        g_loss = -tf.reduce_mean(d_fake)

    grads = tape.gradient(g_loss, G.trainable_variables)
    grads = [tf.clip_by_norm(g, 5.0) if g is not None else None for g in grads]
    g_opt.apply_gradients(zip(grads, G.trainable_variables))
    return g_loss

# -------------------------
# 9) Eval sampling from G logits
# -------------------------
def _apply_temperature_np(p, temperature=1.0):
    p = np.asarray(p, dtype=np.float64)
    p = np.log(p + 1e-12) / float(temperature)
    p = np.exp(p)
    return p / (p.sum() + 1e-12)

def _top_k_top_p_np(p, top_k=0, top_p=1.0):
    p = p.copy()
    if top_k and top_k > 0:
        idx = np.argsort(p)[::-1]
        keep = idx[:top_k]
        mask = np.ones_like(p, dtype=bool)
        mask[keep] = False
        p[mask] = 0.0
    if top_p < 1.0:
        idx = np.argsort(p)[::-1]
        sorted_p = p[idx]
        cumsum = np.cumsum(sorted_p)
        cutoff = cumsum > top_p
        if np.any(cutoff):
            first = int(np.argmax(cutoff))
            remove = idx[first+1:]
            p[remove] = 0.0
    s = p.sum()
    if s <= 0:
        return None
    return p / s

def sample_eval_selfies(num_samples):
    z = tf.random.normal([num_samples, NOISE_DIM])
    logits = G(z, training=False).numpy()

    ids = np.full((num_samples, seq_len), pad_id, dtype=np.int64)
    ids[:, 0] = bos_id

    for t in range(1, seq_len):
        for i in range(num_samples):
            p = tf.nn.softmax(logits[i, t, :]).numpy()
            p = _apply_temperature_np(p, EVAL_TEMPERATURE)
            p = _top_k_top_p_np(p, top_k=EVAL_TOP_K, top_p=EVAL_TOP_P)
            if p is None:
                ids[i, t] = eos_id
            else:
                ids[i, t] = np.random.choice(len(p), p=p)

    selfies_list = [indices_to_selfies(seq) for seq in ids]
    return selfies_list

def eval_samples(tag=""):
    selfies_list = sample_eval_selfies(EVAL_SAMPLES)

    valid_mask = [is_selfies_valid(s) for s in selfies_list]
    valid = [s for s, ok in zip(selfies_list, valid_mask) if ok]
    invalid = [s for s, ok in zip(selfies_list, valid_mask) if not ok]

    validity = 100.0 * len(valid) / len(selfies_list)
    uniq = (100.0 * len(set(valid)) / len(valid)) if valid else 0.0

    print(f"[EVAL {tag}] Total={len(selfies_list)} | Valid={len(valid)} ({validity:.2f}%) | "
          f"Unique(valid)={len(set(valid))}/{len(valid)} ({uniq:.2f}%)")

    safe_tag = str(tag).replace(" ", "_")
    pd.DataFrame({"selfies": selfies_list, "is_valid": valid_mask}).to_csv(
        f"{out_dir}wgan_selfies_stable_{safe_tag}_ALL.csv", index=False
    )
    pd.DataFrame({"selfies": valid}).to_csv(
        f"{out_dir}wgan_selfies_stable_{safe_tag}_VALID.csv", index=False
    )
    pd.DataFrame({"selfies": invalid}).to_csv(
        f"{out_dir}wgan_selfies_stable_{safe_tag}_INVALID.csv", index=False
    )

    return validity, uniq

# -------------------------
# 10) RUN
# -------------------------
it = iter(ds)
start_ep = int(ckpt.epoch.numpy())

print("\n===== Starting WGAN-GP training on SELFIES (stable version) =====")
for ep in range(start_ep, WGAN_EPOCHS):
    frac = ep / max(1, WGAN_EPOCHS - 1)
    tau = (1 - frac) * GUMBEL_TAU_START + frac * GUMBEL_TAU_END

    dloss_last = 0.0
    gloss_last = 0.0

    for step in range(STEPS_PER_EPOCH):
        real_batch = next(it)

        for _ in range(N_CRITIC):
            dloss = critic_step(real_batch, tau=tf.constant(tau, tf.float32))
            dloss_last = float(dloss)

        gloss = generator_step(tau=tf.constant(tau, tf.float32))
        gloss_last = float(gloss)

    ckpt.epoch.assign(ep + 1)
    manager.save()

    if (ep + 1) % EVAL_EVERY == 0:
        validity, uniq = eval_samples(tag=f"ep{ep+1}")

        best_valid = float(ckpt.best_valid.numpy())

        if validity > best_valid + MIN_DELTA:
            ckpt.best_valid.assign(validity)
            ckpt.patience_counter.assign(0)

            G.save_weights(os.path.join(best_dir, "best_generator.weights.h5"))
            meta = {
                "best_epoch": int(ep + 1),
                "best_validity": float(validity),
                "best_uniqueness": float(uniq),
            }
            with open(os.path.join(best_dir, "best_generator_meta.json"), "w") as f:
                json.dump(meta, f, indent=2)

            print(f"✅ New best model saved at epoch {ep+1}")

        elif abs(validity - best_valid) <= MIN_DELTA:
            ckpt.patience_counter.assign(0)
            print("✅ Performance matches best score; patience reset")

        else:
            ckpt.patience_counter.assign(int(ckpt.patience_counter.numpy()) + 1)
            print(f"patience = {int(ckpt.patience_counter.numpy())}/{PATIENCE}")

        print(f"Epoch {ep+1}/{WGAN_EPOCHS} | tau={tau:.3f} | D_loss={dloss_last:.4f} | G_loss={gloss_last:.4f} | ✅ saved")

        if int(ckpt.patience_counter.numpy()) >= PATIENCE:
            print("⛔ Early stopping triggered to avoid collapse")
            break

print("\n✅ Done. Check Drive outputs + checkpoints.")

Mounted at /content/drive
⚠️ No GPU detected
X: (150000, 89) seq_len: 89 vocab: 105
IDs: {'PAD': 0, 'BOS': 1, 'EOS': 2}
✅ Restored: /content/drive/MyDrive/Diss/ckpt_wgan_gp_SELFIES_stable/ckpt-8
✅ Resuming from epoch: 8
✅ Best validity so far: 100.0

===== Starting WGAN-GP training on SELFIES (stable version) =====
[EVAL ep9] Total=300 | Valid=300 (100.00%) | Unique(valid)=300/300 (100.00%)
✅ Performance matches best score; patience reset
Epoch 9/40 | tau=1.138 | D_loss=-0.7431 | G_loss=-7.0517 | ✅ saved
[EVAL ep10] Total=300 | Valid=300 (100.00%) | Unique(valid)=300/300 (100.00%)
✅ Performance matches best score; patience reset
Epoch 10/40 | tau=1.131 | D_loss=-1.2928 | G_loss=-7.5705 | ✅ saved
[EVAL ep11] Total=300 | Valid=300 (100.00%) | Unique(valid)=300/300 (100.00%)
✅ Performance matches best score; patience reset
Epoch 11/40 | tau=1.123 | D_loss=-1.2456 | G_loss=-6.6412 | ✅ saved
[EVAL ep12] Total=300 | Valid=299 (99.67%) | Unique(valid)=299/299 (100.00%)
patience = 1/5
Epoch 12

In [ ]:
# 1) Install Miniconda (Miniforge) and RDKit from conda-forge, then other packages.
#    Run this cell ONCE in a fresh Colab runtime. After it finishes, you'll RESTART the runtime.
#    This is the recommended way to get RDKit working robustly in Colab.

# Download and install Miniforge (lightweight conda)
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O Miniforge3.sh
!bash Miniforge3.sh -b -p /usr/local

# Ensure conda is available
import sys
sys.path.append('/usr/local/lib/python3.10/site-packages')  # adjust if needed by Colab python version

# Add conda to PATH for shell commands in this notebook
import os
os.environ['PATH'] = "/usr/local/bin:" + os.environ['PATH']

# Update conda and install rdkit from conda-forge
# (choose a recent rdkit version available on conda-forge)
!conda install -y -c conda-forge rdkit

# Install other packages via pip (selfies, fcd, scipy, etc.)
!pip install -q selfies scipy fcd

ERROR: File or directory already exists: '/usr/local'
If you want to update an existing installation, use the -u option.
/bin/bash: line 1: conda: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 18.0 MB/s eta 0:00:00


In [ ]:
# بعد Restart runtime شغّلي هذا للتأكد من أن كل المكتبات تعمل:
import sys
print("Python:", sys.version)
import rdkit
from rdkit import Chem
import selfies as sf
print("RDKit and selfies OK — versions:", rdkit.__version__, sf.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
RDKit and selfies OK — versions: 2025.09.6 2.1.1


In [ ]:
# GuacaMol 10k evaluation (resume-proof) for your SeqGenerator (logits-based)
# - Phase A: Validity over 10,000 generated
# - Phase B: Uniqueness: collect 10,000 RDKit-valid (compute uniqueness)
# - Phase C: Novelty/KL/Internal diversity/FCD: collect 10,000 unique canonical
# - Resume-proof: saves progress for each phase, can re-run to continue
# Run in Colab (GPU recommended). Edit USER PATHS below.

!pip -q install rdkit-pypi selfies scipy fcd

import os, time, json, math, random
from collections import Counter
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
import selfies as sf
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, Lipinski
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit import DataStructs
from google.colab import drive
drive.mount('/content/drive')

# -------------------------
# --- USER PATHS ---
# -------------------------
OUT_DIR = "/content/drive/MyDrive/Diss/eval_guacamol_10000"   # output dir for metrics + generated
os.makedirs(OUT_DIR, exist_ok=True)

BEST_WEIGHTS = "/content/drive/MyDrive/Diss/best_wgan_gp_SELFIES_stable/best_generator.weights.h5"
VOCAB_PATH   = "/content/drive/MyDrive/Diss/vocab_selfies_new.csv"
TOKENIZED_PATH = "/content/drive/MyDrive/Diss/tokenized_selfies_new.csv"
TRAIN_REF_PATH = "/content/drive/MyDrive/Diss/selfies_tokens_preview.csv"  # must contain 'smiles' column

# generation params (tune if needed)
NOISE_DIM = 128
BATCH_GEN = 200
GEN_TEMPERATURE = 0.9
GEN_TOP_K = 60
GEN_TOP_P = 0.92
SEED = 42
TIMEOUT_HOURS = 8

# GuacaMol targets
TARGET_GEN = 10000      # for validity: generate this many samples
TARGET_VALID = 10000    # for uniqueness: collect this many RDKit-valid
TARGET_UNIQUE = 10000   # for novelty/kl/etc: collect this many unique canonical

# Save filenames (progress)
PHASE_A_FILE = os.path.join(OUT_DIR, "phaseA_validity_progress.csv")   # stores generated smiles + is_valid
PHASE_B_FILE = os.path.join(OUT_DIR, "phaseB_valids_canonical.csv")    # stores canonical valid list (order preserved)
PHASE_C_FILE = os.path.join(OUT_DIR, "phaseC_unique_canonical.csv")    # stores unique canonical collected
METRICS_FILE = os.path.join(OUT_DIR, "guacamol_metrics_10000.json")

# -------------------------
# Utilities
# -------------------------
def _apply_temperature_np(p, temperature=1.0):
    p = np.asarray(p, dtype=np.float64)
    p = np.log(p + 1e-12) / float(temperature)
    p = np.exp(p)
    return p / (p.sum() + 1e-12)

def _top_k_top_p_np(p, top_k=0, top_p=1.0):
    p = p.copy()
    if top_k and top_k > 0:
        idx = np.argsort(p)[::-1]
        keep = idx[:top_k]
        mask = np.ones_like(p, dtype=bool)
        mask[keep] = False
        p[mask] = 0.0
    if top_p < 1.0:
        idx = np.argsort(p)[::-1]
        sorted_p = p[idx]
        cumsum = np.cumsum(sorted_p)
        cutoff = cumsum > top_p
        if np.any(cutoff):
            first = int(np.argmax(cutoff))
            remove = idx[first+1:]
            p[remove] = 0.0
    s = p.sum()
    if s <= 0:
        return None
    return p / s

def selfies_to_smiles(selfies_str):
    try:
        return sf.decoder(str(selfies_str))
    except Exception:
        return ""

def canon_smiles(sm):
    try:
        m = Chem.MolFromSmiles(sm)
        if m is None:
            return None
        return Chem.MolToSmiles(m, canonical=True)
    except Exception:
        return None

# -------------------------
# Load vocab + seq_len
# -------------------------
vocab_df = pd.read_csv(VOCAB_PATH).sort_values("index").reset_index(drop=True)
idx2token = dict(zip(vocab_df['index'].astype(int), vocab_df['token'].astype(str)))
token2idx = dict(zip(vocab_df['token'].astype(str), vocab_df['index'].astype(int)))

PAD_TOKEN, BOS_TOKEN, EOS_TOKEN = "<pad>", "<bos>", "<eos>"
pad_id = int(token2idx[PAD_TOKEN])
bos_id = int(token2idx[BOS_TOKEN])
eos_id = int(token2idx[EOS_TOKEN])
vocab_size = len(token2idx)
seq_len = pd.read_csv(TOKENIZED_PATH, nrows=5).shape[1]

print("vocab_size:", vocab_size, "seq_len:", seq_len, "pad/bos/eos:", pad_id, bos_id, eos_id)

def indices_to_selfies(seq):
    out = []
    for idx in seq:
        idx = int(idx)
        if idx in (pad_id, eos_id):
            break
        if idx == bos_id:
            continue
        out.append(idx2token.get(idx, ""))
    return "".join(out)

# -------------------------
# Recreate SeqGenerator (must match training)
# -------------------------
class SeqGenerator(Model):
    def __init__(self, vocab_size, noise_dim=128, hidden=256):
        super().__init__()
        self.fc = layers.Dense(hidden, activation="relu")
        self.rep = layers.RepeatVector(seq_len)
        self.gru = layers.GRU(hidden, return_sequences=True)
        self.out = layers.Dense(vocab_size)
    def call(self, z, training=False):
        x = self.fc(z)
        x = self.rep(x)
        x = self.gru(x, training=training)
        return self.out(x)

G = SeqGenerator(vocab_size, noise_dim=NOISE_DIM, hidden=256)
_ = G(tf.zeros((2, NOISE_DIM), dtype=tf.float32), training=False)
if not os.path.exists(BEST_WEIGHTS):
    raise FileNotFoundError(f"Best weights not found: {BEST_WEIGHTS}")
G.load_weights(BEST_WEIGHTS)
print("Loaded best weights:", BEST_WEIGHTS)

# -------------------------
# Sampling function (logits-based)
# -------------------------
def generate_batch_selfies(batch_size, temperature=GEN_TEMPERATURE, top_k=GEN_TOP_K, top_p=GEN_TOP_P):
    z = tf.random.normal([batch_size, NOISE_DIM])
    logits = G(z, training=False).numpy()   # (B, T, V)
    ids = np.full((batch_size, seq_len), pad_id, dtype=np.int64)
    ids[:, 0] = bos_id
    for t in range(1, seq_len):
        for i in range(batch_size):
            p = tf.nn.softmax(logits[i, t, :]).numpy()
            p = _apply_temperature_np(p, temperature)
            p = _top_k_top_p_np(p, top_k=top_k, top_p=top_p)
            if p is None:
                ids[i, t] = eos_id
            else:
                ids[i, t] = np.random.choice(len(p), p=p)
    selfies_list = [indices_to_selfies(seq) for seq in ids]
    return selfies_list

# -------------------------
# Phase A: Validity over 10k generated
# -------------------------
def phase_a_generate(target=TARGET_GEN, batch=BATCH_GEN, resume=True):
    # resume: load PHASE_A_FILE if exists
    if resume and os.path.exists(PHASE_A_FILE):
        df = pd.read_csv(PHASE_A_FILE)
        total = len(df)
        print("Resuming Phase A: already generated", total)
    else:
        df = pd.DataFrame(columns=["selfies", "smiles", "is_rdkit_valid"])
        total = 0

    start = time.time()
    while total < target:
        need = min(batch, target - total)
        batch_selfies = generate_batch_selfies(need)
        rows = []
        for s in batch_selfies:
            sm = selfies_to_smiles(s)
            is_valid = False
            if sm:
                is_valid = Chem.MolFromSmiles(sm) is not None
            rows.append({"selfies": s, "smiles": sm if sm else "", "is_rdkit_valid": bool(is_valid)})
        df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
        total = len(df)
        # save progress every batch
        df.to_csv(PHASE_A_FILE, index=False)
        if total % 2000 == 0 or total == target:
            valid_count = df["is_rdkit_valid"].astype(bool).sum()
            print(f"Phase A: generated {total}, valid so far: {valid_count}")
        # timeout
        if (time.time() - start) > TIMEOUT_HOURS * 3600:
            print("Phase A timeout reached")
            break

    # compute validity over the generated set (as GuacaMol: valid / 10000)
    valid_count = int(df["is_rdkit_valid"].astype(bool).sum())
    validity_pct = 100.0 * valid_count / max(1, len(df))
    print("Phase A completed: generated", len(df), "valid:", valid_count, "validity%:", validity_pct)
    return df, validity_pct

# -------------------------
# Phase B: collect 10k RDKit-valid (for uniqueness)
# -------------------------
def phase_b_collect(target_valid=TARGET_VALID, batch=BATCH_GEN, resume=True):
    # resume if file exists
    collected = []
    total_generated = 0
    if resume and os.path.exists(PHASE_B_FILE):
        prev = pd.read_csv(PHASE_B_FILE)
        collected = prev["canonical_smiles"].dropna().astype(str).tolist()
        total_generated = int(prev.attrs.get("total_generated", 0)) if "total_generated" in prev.attrs else len(collected)
        print("Resuming Phase B: collected", len(collected), "valids")
    else:
        prev = None

    start = time.time()
    while len(collected) < target_valid:
        batch_selfies = generate_batch_selfies(batch)
        total_generated += len(batch_selfies)
        for s in batch_selfies:
            sm = selfies_to_smiles(s)
            if not sm:
                continue
            c = canon_smiles(sm)
            if c is None:
                continue
            collected.append(c)
            if len(collected) >= target_valid:
                break
        # persist every batch
        pd.DataFrame({"canonical_smiles": collected}).to_csv(PHASE_B_FILE, index=False)
        # store total_generated in a small json alongside
        with open(PHASE_B_FILE + ".meta.json", "w") as f:
            json.dump({"total_generated": total_generated}, f)
        if total_generated % 2000 == 0 or len(collected) % 1000 == 0:
            print(f"Phase B: total_generated {total_generated}, collected valid: {len(collected)}")
        if (time.time() - start) > TIMEOUT_HOURS * 3600:
            print("Phase B timeout reached")
            break

    unique_count = len(set(collected))
    uniqueness_pct = 100.0 * unique_count / max(1, len(collected))
    print(f"Phase B completed: collected {len(collected)} valids, uniqueness% = {uniqueness_pct:.2f}")
    return collected, total_generated, uniqueness_pct

# -------------------------
# Phase C: collect 10k unique canonical (for novelty/KL etc.)
# -------------------------
def phase_c_collect(target_unique=TARGET_UNIQUE, batch=BATCH_GEN, resume=True):
    unique_list = []
    seen = set()
    total_generated = 0
    if resume and os.path.exists(PHASE_C_FILE):
        prev = pd.read_csv(PHASE_C_FILE)
        unique_list = prev["canonical_smiles"].dropna().astype(str).tolist()
        seen = set(unique_list)
        total_generated = int(prev.attrs.get("total_generated", 0)) if "total_generated" in prev.attrs else len(unique_list)
        print("Resuming Phase C: already unique collected:", len(unique_list))
    start = time.time()
    while len(unique_list) < target_unique:
        batch_selfies = generate_batch_selfies(batch)
        total_generated += len(batch_selfies)
        for s in batch_selfies:
            sm = selfies_to_smiles(s)
            if not sm:
                continue
            c = canon_smiles(sm)
            if c is None:
                continue
            if c not in seen:
                seen.add(c)
                unique_list.append(c)
                if len(unique_list) >= target_unique:
                    break
        # save progress
        pd.DataFrame({"canonical_smiles": unique_list}).to_csv(PHASE_C_FILE, index=False)
        with open(PHASE_C_FILE + ".meta.json", "w") as f:
            json.dump({"total_generated": total_generated}, f)
        if total_generated % 2000 == 0 or len(unique_list) % 1000 == 0:
            print(f"Phase C: total_generated {total_generated}, unique_collected: {len(unique_list)}")
        if (time.time() - start) > TIMEOUT_HOURS * 3600:
            print("Phase C timeout reached")
            break
    print("Phase C completed: unique collected:", len(unique_list))
    return unique_list, total_generated

# -------------------------
# Metrics computation (after having unique_set and train set)
# -------------------------
def featurize(smiles_list):
    rows = []
    for s in smiles_list:
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        rows.append({
            "MW": Descriptors.MolWt(m),
            "LogP": Crippen.MolLogP(m),
            "TPSA": rdMolDescriptors.CalcTPSA(m),
            "HBD": Lipinski.NumHDonors(m),
            "HBA": Lipinski.NumHAcceptors(m),
            "RotB": Lipinski.NumRotatableBonds(m),
            "Rings": Lipinski.RingCount(m),
        })
    return pd.DataFrame(rows)

def kl_hist(p_samples, q_samples, bins=50, eps=1e-8):
    p = np.asarray(p_samples); q = np.asarray(q_samples)
    if len(p) == 0 or len(q) == 0:
        return np.nan
    lo = min(p.min(), q.min()); hi = max(p.max(), q.max())
    if lo == hi:
        return 0.0
    P, _ = np.histogram(p, bins=bins, range=(lo, hi), density=True)
    Q, _ = np.histogram(q, bins=bins, range=(lo, hi), density=True)
    P = P + eps; Q = Q + eps
    P = P / P.sum(); Q = Q / Q.sum()
    return float(np.sum(P * np.log(P / Q)))

def compute_metrics(unique_set, total_generated_for_unique):
    # prepare training canonical set
    train_df = pd.read_csv(TRAIN_REF_PATH)
    smiles_col = None
    for c in ["smiles", "SMILES", "canonical_smiles", "canon_smiles"]:
        if c in train_df.columns:
            smiles_col = c
            break
    if smiles_col is None:
        raise ValueError("Train reference must contain a SMILES column")
    train_smiles_all = train_df[smiles_col].astype(str).str.strip()
    train_canon = []
    for sm in train_smiles_all:
        c = canon_smiles(sm)
        if c:
            train_canon.append(c)
    # Novelty
    train_set = set(train_canon)
    novel = [c for c in unique_set if c not in train_set]
    novelty_pct = 100.0 * len(novel) / max(1, len(unique_set))
    # Featurize
    train_feat = featurize(train_canon)
    gen_feat = featurize(unique_set)
    props = ["MW","LogP","TPSA","HBD","HBA","RotB","Rings"]
    kl_per_prop = {}
    for p in props:
        try:
            kl_per_prop[p] = kl_hist(train_feat[p].values, gen_feat[p].values, bins=50)
        except Exception:
            kl_per_prop[p] = np.nan
    # GuacaMol aggregation: mean of exp(-DKL_i)
    import math
    valid_kl_vals = [v for v in kl_per_prop.values() if not math.isnan(v)]
    mean_exp_neg = float(np.mean([math.exp(-v) for v in valid_kl_vals])) if len(valid_kl_vals)>0 else np.nan
    kl_mean = float(np.nanmean(valid_kl_vals)) if len(valid_kl_vals)>0 else np.nan
    # internal diversity approx
    fpgen = GetMorganGenerator(radius=2, fpSize=2048)
    mols = [Chem.MolFromSmiles(s) for s in unique_set]
    mols = [m for m in mols if m is not None]
    fps = [fpgen.GetFingerprint(m) for m in mols]
    internal_div = None
    if len(fps) < 2:
        internal_div = np.nan
    else:
        rng = np.random.default_rng(SEED)
        n = len(fps)
        pairs = min(2000, n*(n-1)//2)
        sims = []
        for _ in range(pairs):
            i = int(rng.integers(0,n))
            j = int(rng.integers(0,n-1))
            if j >= i:
                j += 1
            sims.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
        internal_div = float(1.0 - np.mean(sims))
    # FCD (optional)
    fcd_score = None
    try:
        from fcd import get_fcd
        import numpy.random as npr
        rng = npr.default_rng(SEED)
        train_sample = list(rng.choice(train_canon, size=min(10000, len(train_canon)), replace=False))
        fcd_score = float(get_fcd(train_sample, unique_set))
    except Exception as e:
        fcd_score = None
    metrics = {
        "novelty_pct": novelty_pct,
        "kl_per_prop": kl_per_prop,
        "kl_mean": kl_mean,
        "guacamol_kl_mean_exp_neg": mean_exp_neg,
        "internal_diversity": internal_div,
        "fcd": fcd_score,
        "total_generated_for_unique": int(total_generated_for_unique),
    }
    return metrics

# -------------------------
# Run whole pipeline with resume
# -------------------------
if __name__ == "__main__":
    # Phase A
    print(">>> START PHASE A (validity over 10k generation)")
    df_phase_a, validity_pct = phase_a_generate(TARGET_GEN, BATCH_GEN, resume=True)

    # Phase B
    print("\n>>> START PHASE B (collect 10k RDKit-valid for uniqueness)")
    collected_valids, total_generated_b, uniqueness_pct = phase_b_collect(TARGET_VALID, BATCH_GEN, resume=True)

    # Phase C
    print("\n>>> START PHASE C (collect 10k unique canonical for novelty/KL/... )")
    unique_set, total_generated_c = phase_c_collect(TARGET_UNIQUE, BATCH_GEN, resume=True)

    # Compute distribution metrics
    print("\n>>> COMPUTE DISTRIBUTION METRICS")
    dist_metrics = compute_metrics(unique_set, total_generated_c)

    # Save summary metrics
    metrics = {
        "validity_10k_pct": float(validity_pct),
        "total_generated_for_validity": int(len(df_phase_a)),
        "uniqueness_pct_on_10k_valid": float(uniqueness_pct),
        "total_generated_for_10k_valid": int(total_generated_b),
        "collected_unique_10k_count": int(len(unique_set)),
    }
    metrics.update(dist_metrics)
    with open(METRICS_FILE, "w") as f:
        json.dump(metrics, f, indent=2)

    # Save final generated sets (for reproducibility)
    pd.DataFrame({"selfies": df_phase_a["selfies"], "smiles": df_phase_a["smiles"], "is_rdkit_valid": df_phase_a["is_rdkit_valid"]}).to_csv(os.path.join(OUT_DIR, "phaseA_all_generated.csv"), index=False)
    pd.DataFrame({"canonical_smiles_10k_valid": collected_valids[:TARGET_VALID]}).to_csv(os.path.join(OUT_DIR, "gen_10k_valid_canonical.csv"), index=False)
    pd.DataFrame({"canonical_smiles_10k_unique": unique_set[:TARGET_UNIQUE]}).to_csv(os.path.join(OUT_DIR, "gen_10k_unique_canonical.csv"), index=False)

    print("\nDONE. Metrics saved to:", METRICS_FILE)
    print(json.dumps(metrics, indent=2))

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi
Mounted at /content/drive
vocab_size: 105 seq_len: 89 pad/bos/eos: 0 1 2
Loaded best weights: /content/drive/MyDrive/Diss/best_wgan_gp_SELFIES_stable/best_generator.weights.h5
>>> START PHASE A (validity over 10k generation)
Phase A: generated 2000, valid so far: 1955
Phase A: generated 4000, valid so far: 3916
Phase A: generated 6000, valid so far: 5858
Phase A: generated 8000, valid so far: 7809
Phase A: generated 10000, valid so far: 9758
Phase A completed: generated 10000 valid: 9758 validity%: 97.58

>>> START PHASE B (collect 10k RDKit-valid for uniqueness)
Phase B: total_generated 2000, collected valid: 1943
Phase B: total_generated 4000, collected valid: 3886
Phase B: total_generated 6000, collected valid: 5819
Phase B: total_generated 8000, collected valid: 7770
Phase B: total_generated 10000, collected valid: 9718
Phase B: total_

In [ ]:
# RUN3: automated 3 runs with different seeds, aggregate metrics (run AFTER your main script)
import os, json, math
import pandas as pd
import numpy as np
import random
import tensorflow as tf

# list of seeds and base outdir
seeds = [42, 101, 777]   # غيّري إذا بدك
base_outdir = "/content/drive/MyDrive/Diss/eval_guacamol_10000_runs"

# ensure the run functions are available in the notebook (they are if you ran the big script)
# We will call: phase_a_generate, phase_b_collect, phase_c_collect, compute_metrics

rows = []

for seed in seeds:
    print("\n\n========================")
    print("Starting run with seed:", seed)
    print("========================\n")
    # set randomness for reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # per-run outdir
    outdir = os.path.join(base_outdir, f"seed_{seed}")
    os.makedirs(outdir, exist_ok=True)

    # override global OUT_DIR and progressive filenames used by functions
    # PHASE_A_FILE, PHASE_B_FILE, PHASE_C_FILE, METRICS_FILE are globals in your script
    global OUT_DIR, PHASE_A_FILE, PHASE_B_FILE, PHASE_C_FILE, METRICS_FILE
    OUT_DIR = outdir
    PHASE_A_FILE = os.path.join(OUT_DIR, "phaseA_validity_progress.csv")
    PHASE_B_FILE = os.path.join(OUT_DIR, "phaseB_valids_canonical.csv")
    PHASE_C_FILE = os.path.join(OUT_DIR, "phaseC_unique_canonical.csv")
    METRICS_FILE = os.path.join(OUT_DIR, "guacamol_metrics_10000.json")

    # also ensure target constants (optional)
    # TARGET_GEN, TARGET_VALID, TARGET_UNIQUE, BATCH_GEN etc. are global; keep defaults or override here.

    # Run phases (resume=False to force fresh run; set True if you want to resume)
    # If you previously ran and want fresh results, remove old outdir or set resume=False inside functions
    df_phase_a, validity_pct = phase_a_generate(TARGET_GEN, BATCH_GEN, resume=False)
    collected_valids, total_generated_b, uniqueness_pct = phase_b_collect(TARGET_VALID, BATCH_GEN, resume=False)
    unique_set, total_generated_c = phase_c_collect(TARGET_UNIQUE, BATCH_GEN, resume=False)

    # compute metrics
    dist_metrics = compute_metrics(unique_set, total_generated_c)

    # build summary row (flatten relevant metrics)
    row = {
        "seed": seed,
        "validity_10k_pct": float(validity_pct),
        "total_generated_for_validity": int(len(df_phase_a)),
        "uniqueness_pct_on_10k_valid": float(uniqueness_pct),
        "total_generated_for_10k_valid": int(total_generated_b),
        "collected_unique_10k_count": int(len(unique_set)),
        "total_generated_for_unique": int(dist_metrics.get("total_generated_for_unique", total_generated_c) if isinstance(dist_metrics, dict) else int(total_generated_c)),
        "novelty_pct": float(dist_metrics.get("novelty_pct", np.nan)),
        "kl_mean": float(dist_metrics.get("kl_mean", np.nan)),
        "guacamol_kl_mean_exp_neg": float(dist_metrics.get("guacamol_kl_mean_exp_neg", np.nan)),
        "internal_diversity": float(dist_metrics.get("internal_diversity", np.nan)),
        "fcd": float(dist_metrics.get("fcd", np.nan))
    }

    # also flatten kl_per_prop into columns (if exists)
    kl = dist_metrics.get("kl_per_prop", {}) if isinstance(dist_metrics, dict) else {}
    for prop in ["MW","LogP","TPSA","HBD","HBA","RotB","Rings"]:
        row[f"kl_{prop}"] = float(kl.get(prop, np.nan))

    # save per-run metrics as JSON too
    with open(os.path.join(outdir, "summary_metrics.json"), "w") as f:
        json.dump(row, f, indent=2)

    rows.append(row)

# create DataFrame of runs
df_runs = pd.DataFrame(rows)
summary_csv = os.path.join(base_outdir, "metrics_3runs.csv")
df_runs.to_csv(summary_csv, index=False)
print("\nSaved per-run metrics to:", summary_csv)

# compute mean and std for numeric columns
numeric_cols = [c for c in df_runs.columns if c not in ("seed",)]
means = df_runs[numeric_cols].mean()
stds  = df_runs[numeric_cols].std(ddof=1)  # sample std

summary = pd.DataFrame({"mean": means, "std": stds})
summary_csv = os.path.join(base_outdir, "metrics_3runs_summary.csv")
summary.to_csv(summary_csv)
print("Saved aggregated mean/std to:", summary_csv)

print("\n=== Mean and Std ===")
print(summary.round(4))



Starting run with seed: 42

Phase A: generated 2000, valid so far: 1926
Phase A: generated 4000, valid so far: 3873
Phase A: generated 6000, valid so far: 5814
Phase A: generated 8000, valid so far: 7753
Phase A: generated 10000, valid so far: 9695
Phase A completed: generated 10000 valid: 9695 validity%: 96.95
Phase B: total_generated 2000, collected valid: 1933
Phase B: total_generated 4000, collected valid: 3892
Phase B: total_generated 6000, collected valid: 5835
Phase B: total_generated 8000, collected valid: 7771
Phase B: total_generated 10000, collected valid: 9727
Phase B: total_generated 10400, collected valid: 10000
Phase B completed: collected 10000 valids, uniqueness% = 92.95
Phase C: total_generated 2000, unique_collected: 1881
Phase C: total_generated 4000, unique_collected: 3683
Phase C: total_generated 6000, unique_collected: 5498
Phase C: total_generated 8000, unique_collected: 7336
Phase C: total_generated 10000, unique_collected: 9111
Phase C: total_generated 11200

In [ ]:
import numpy as np

def bootstrap_mean_ci(arr, n_boot=5000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    boots = [np.mean(rng.choice(arr, size=len(arr), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boots, (1-ci)/2*100)
    hi = np.percentile(boots, (1+ci)/2*100)
    return np.mean(arr), np.std(arr, ddof=1), lo, hi

# مثال: arr = [96.95, 97.45, 97.34]  # validity runs

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# افترض الملف metrics_3runs.csv موجود وتحتوي الأعمدة
df = pd.read_csv("/content/drive/MyDrive/Diss/eval_guacamol_10000_runs/metrics_3runs.csv")

metrics = ["validity_10k_pct","uniqueness_pct_on_10k_valid","novelty_pct",
           "guacamol_kl_mean_exp_neg","kl_mean","internal_diversity","fcd"]

# أدخلي أرقام baselines من GuacaMol هنا
baselines = {
  "validity_10k_pct": 95.0,
  "uniqueness_pct_on_10k_valid": 90.0,
  "novelty_pct": 85.0,
  "guacamol_kl_mean_exp_neg": 0.45,
  "kl_mean": 0.5,
  "internal_diversity": 0.80,
  "fcd": 20.0
}

rows=[]
for m in metrics:
    arr = df[m].dropna().values
    mean = arr.mean(); std = arr.std(ddof=1)
    # bootstrap CI
    mean_b, std_b, lo, hi = bootstrap_mean_ci(arr, n_boot=5000, seed=42)
    baseline = baselines.get(m, np.nan)
    baseline_in_CI = (baseline >= lo) and (baseline <= hi) if not np.isnan(baseline) else None
    # small-sample t-test (caveat)
    tstat, pval = stats.ttest_1samp(arr, popmean=baseline) if not np.isnan(baseline) else (np.nan, np.nan)
    rows.append({"metric":m, "mean":mean, "std":std, "ci_lo":lo, "ci_hi":hi,
                 "baseline":baseline, "baseline_in_CI":baseline_in_CI, "tstat":tstat, "pval":pval})
comp = pd.DataFrame(rows).set_index("metric")
print(comp.round(4))
comp.to_csv("/content/drive/MyDrive/Diss/eval_guacamol_10000_runs/comparison_with_baselines.csv")

                                mean     std    ci_lo    ci_hi  baseline  \
metric                                                                     
validity_10k_pct             97.2467  0.2627  96.9500  97.4500     95.00   
uniqueness_pct_on_10k_valid  92.9567  0.2601  92.7000  93.2200     90.00   
novelty_pct                  99.2133  0.0058  99.2100  99.2200     85.00   
guacamol_kl_mean_exp_neg      0.3290  0.0105   0.3204   0.3407      0.45   
kl_mean                       1.7765  0.0823   1.6872   1.8491      0.50   
internal_diversity            0.9055  0.0006   0.9049   0.9060      0.80   
fcd                          40.1398  0.0652  40.0648  40.1835     20.00   

                             baseline_in_CI      tstat    pval  
metric                                                          
validity_10k_pct                      False    14.8105  0.0045  
uniqueness_pct_on_10k_valid           False    19.6917  0.0026  
novelty_pct                           False  4264.0000 